# 💾 Lesson 14 · Checkpoint + Store 持久化

一句话分清两者：

| | **Checkpoint（检查点）** | **Store（长期记忆）** |
|---|---|---|
| 像什么 | 聊天会话的「存档」 | 独立的「记事本 / 知识库」 |
| 存什么 | 图的运行状态（消息、中间变量） | 你自己定义的数据（偏好、事实等） |
| 范围 | 跟着 `thread_id`，一条会话一条线 | 可跨多条会话共享 |
| 寿命 | 偏短期：续聊、回放、容错 | 偏长期：记住用户是谁、喜欢什么 |

**Checkpoint（Checkpointer）**  
把某一条线程（`thread_id`）里图的状态存成检查点。  
适合：多轮对话不断档、人机协作中途暂停、时光旅行（回到某一步）、程序崩了还能接着跑。

**Store**  
把业务数据存在图状态之外。  
适合：用户偏好、长期事实、多会话共用的知识——换一个 `thread_id` 也能读到。

> 口诀：**Checkpoint 管「这次对话怎么跑到现在」；Store 管「我该长期记住什么」。**


## 逻辑总览

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart LR
    S([START]) --> J[generate_joke]
    J --> X[generate_explanation]
    X --> E([END])

    CP[(InMemorySaver<br/>thread_id)] -.-> J
    CP -.-> X


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class S input
    class J,X llm
    class CP tool
    class E output
```

**要点：** 没有 Checkpoint 就没有可靠多轮会话；`thread_id` 是会话钥匙。



# Imports

注意引入 `InMemorySaver`（内存版 Checkpointer）和 `InMemoryStore`（跨线程长期记忆）。


In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore


C:\Users\86137\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\triton\windows_utils.py:372: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


In [2]:
load_dotenv()

True

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

SYSTEM_PROMPT = (
    "Always reply in the same language the user uses. "
    "If the user writes in Chinese, answer in Chinese; "
    "if in English, answer in English; match other languages the same way. "
    "Do not switch or translate unless the user explicitly asks."
)

llm = ChatOpenAI(
    model="deepseek-v4-pro",
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    timeout=120,
    max_tokens=1200,
    max_retries=1,
    extra_body={"thinking": {"type": "disabled"}},
)


def chat(user_text: str) -> str:
    """带 system prompt 调用模型。"""
    return llm.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_text),
        ]
    ).content


# 定义 State


In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

# Node — generate_joke


In [6]:
def generate_joke(state: JokeState, config: RunnableConfig, *, store: BaseStore):
    """生成笑话，并把结果写入 Store（跨线程可读）。"""
    prompt = (
        f"Topic: {state['topic']}\n"
        "Write a joke about this topic. "
        "Reply in the same language as the topic."
    )
    response = chat(prompt)

    # Store：按 (namespace, key) 存长期记忆
    thread_id = config["configurable"]["thread_id"]
    store.put(
        ("jokes", thread_id),  # namespace
        state["topic"],        # key
        {"topic": state["topic"], "joke": response},
    )
    return {"joke": response}

# Node — generate_explanation


In [7]:
def generate_explanation(state: JokeState, config: RunnableConfig, *, store: BaseStore):
    """生成解释，并更新 Store 中同一 topic 的记录。"""
    prompt = (
        f"Joke:\n{state['joke']}\n\n"
        "Write an explanation for this joke. "
        "Reply in the same language as the joke."
    )
    response = chat(prompt)

    thread_id = config["configurable"]["thread_id"]
    prev = store.get(("jokes", thread_id), state["topic"])
    payload = dict(prev.value) if prev else {"topic": state["topic"], "joke": state["joke"]}
    payload["explanation"] = response
    store.put(("jokes", thread_id), state["topic"], payload)

    return {"explanation": response}

# 编译图：同时挂上 Checkpointer + Store


## 逻辑总览

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart LR
    S([START]) --> J[generate_joke]
    J --> X[generate_explanation]
    X --> E([END])

    CP[(InMemorySaver<br/>thread_id)] -.-> J
    CP -.-> X


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class S input
    class J,X llm
    class CP tool
    class E output
```

**要点：** 没有 Checkpoint 就没有可靠多轮会话；`thread_id` 是会话钥匙。

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()  # 线程内图状态（随 thread_id）
store = InMemoryStore()         # 跨线程长期记忆（namespace + key）

workflow = graph.compile(checkpointer=checkpointer, store=store)

# 用 thread_id 执行

同一 `thread_id` = 同一会话；换 id 就是新线程。


In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'足球'}, config=config1)

{'topic': '足球',
 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！',
 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}

# 查看历史 Checkpoint

`get_state_history` 列出该线程的状态时间线。


In [10]:
# 方式 1：转成 list（默认新→旧，reversed 后按旧→新倒序打印）
history = list(workflow.get_state_history(config1))
for i, snap in enumerate(reversed(history)):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=-1 next=['__start__'] ---
{}
--- history[1] step=0 next=['generate_joke'] ---
{'topic': '足球'}
--- history[2] step=1 next=['generate_explanation'] ---
{'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！'}
--- history[3] step=2 next=[] ---
{'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！', 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}


In [12]:
# 当前 checkpoint 快照（必须带 config）
snap = workflow.get_state(config1)
print("values:", snap.values)
print()
print("next:", snap.next)
print()
print("checkpoint_id:", snap.config["configurable"].get("checkpoint_id"))

values: {'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！', 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}

next: ()

checkpoint_id: 1f19311b-d772-6166-8002-403534e44f36


In [13]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'美少女'}, config=config1)

{'topic': '美少女',
 'joke': '为什么美少女拍照时总爱歪头？  \n因为她们怕“正”脸会变成“政”脸——严肃到能开新闻发布会！',
 'explanation': '这个笑话利用了中文里的谐音梗。“正脸”指的是正面朝向镜头的脸，而“政脸”是生造词，谐音“正脸”，但“政”字让人联想到政治人物在新闻发布会上那种严肃、正式的表情。美少女歪头拍照是为了避免自己的脸看起来太正经、太严肃，就像在开新闻发布会一样，从而保持可爱俏皮的形象。'}

In [14]:
workflow.get_state(config1)

StateSnapshot(values={'topic': '美少女', 'joke': '为什么美少女拍照时总爱歪头？  \n因为她们怕“正”脸会变成“政”脸——严肃到能开新闻发布会！', 'explanation': '这个笑话利用了中文里的谐音梗。“正脸”指的是正面朝向镜头的脸，而“政脸”是生造词，谐音“正脸”，但“政”字让人联想到政治人物在新闻发布会上那种严肃、正式的表情。美少女歪头拍照是为了避免自己的脸看起来太正经、太严肃，就像在开新闻发布会一样，从而保持可爱俏皮的形象。'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f193121-c9a7-635a-8006-76ba0157de04'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-08-08T10:15:42.377455+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f193121-af9d-6fa8-8005-533eb488457a'}}, tasks=(), interrupts=())

In [15]:
# 方式 1：转成 list（默认新→旧，reversed 后按旧→新倒序打印）
history = list(workflow.get_state_history(config1))
for i, snap in enumerate(reversed(history)):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=-1 next=['__start__'] ---
{}
--- history[1] step=0 next=['generate_joke'] ---
{'topic': '足球'}
--- history[2] step=1 next=['generate_explanation'] ---
{'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！'}
--- history[3] step=2 next=[] ---
{'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！', 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}
--- history[4] step=3 next=['__start__'] ---
{'topic': '足球', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！', 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}
--- history[5] step=4 next=['generate_joke'] ---
{'topic': '美少女', 'joke': '为什么足球运动员从来不去银行？\n因为他们怕被“罚点球”！', 'explanation': '这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。'}
--- history[6] step=5 next=['generate_explanation'

In [18]:
workflow.invoke({'topic':'美少男'}, config=config1)
workflow.invoke({'topic':'电风扇'}, config=config1)

{'topic': '电风扇',
 'joke': '电风扇对空调说：“兄弟，你虽然能制冷，但我比你更懂人情世故——我吹出来的风，至少不会让人感冒还赖我头上！”',
 'explanation': '这个笑话利用了拟人手法，把电风扇和空调比作两个在聊天的朋友。电风扇调侃空调虽然制冷能力强，但自己更“懂人情世故”。笑点在于：空调吹出的冷风如果让人着凉感冒，人们往往会怪空调温度太低；而电风扇吹的风相对温和，即使吹久了不舒服，人们也很少直接责怪电风扇。电风扇借此自嘲式地炫耀自己“不背锅”的处世智慧，幽默地揭示了生活中人们对不同电器“责任归属”的双重标准。'}

# 查看 Store

Checkpoint = 某条 `thread_id` 的图状态时间线；Store = 跨线程的 `(namespace, key)` 记忆。

节点里已用 `store.put(("jokes", thread_id), topic, {...})` 写入。

In [20]:
# 逐条打印 Store 全部内容
idx = 0
for ns in store.list_namespaces():
    for item in store.search(ns):
        idx += 1
        print(f"========== store[{idx}] ==========")
        print(f"namespace : {ns}")
        print(f"key       : {item.key}")
        print("value     :")
        for k, v in (item.value or {}).items():
            print(f"  [{k}]")
            print(f"  {v}")
            print()
        print()

if idx == 0:
    print("(store 为空 — 请先重新 Run 编译 + invoke)")

========== store[1] ==========
namespace : ('jokes', '1')
key       : 足球
value     :
  [topic]
  足球

  [joke]
  为什么足球运动员从来不去银行？
因为他们怕被“罚点球”！

  [explanation]
  这个笑话利用了中文里的双关语。“罚点球”在足球比赛中是指因犯规被判罚点球，而“点球”听起来像“点钱”，也就是数钱的意思。所以，足球运动员不去银行，是因为他们怕被“罚点球”——既怕在球场上被判点球，又怕在银行里被罚数钱，形成了一种幽默的文字游戏。


========== store[2] ==========
namespace : ('jokes', '1')
key       : 美少女
value     :
  [topic]
  美少女

  [joke]
  为什么美少女拍照时总爱歪头？  
因为她们怕“正”脸会变成“政”脸——严肃到能开新闻发布会！

  [explanation]
  这个笑话利用了中文里的谐音梗。“正脸”指的是正面朝向镜头的脸，而“政脸”是生造词，谐音“正脸”，但“政”字让人联想到政治人物在新闻发布会上那种严肃、正式的表情。美少女歪头拍照是为了避免自己的脸看起来太正经、太严肃，就像在开新闻发布会一样，从而保持可爱俏皮的形象。


========== store[3] ==========
namespace : ('jokes', '1')
key       : 美少男
value     :
  [topic]
  美少男

  [joke]
  为什么美少男从来不用闹钟？  
因为他们的美貌已经足够“醒”目了！

  [explanation]
  这个笑话利用了中文里的双关和谐音梗。“醒目”这个词通常有两个意思：一是字面意义上的“引人注目、显眼”，二是粤语/口语里常用来形容人聪明、机灵。笑话里说美少男不用闹钟，是因为他们的美貌已经足够“醒”目——这里把“醒目”拆解成“醒”和“目”，玩了一个谐音联想：闹钟是用来“叫醒”人的，而他们的美貌本身就“醒”目，既表示“亮眼到能把人看醒”，又暗指“醒”就是“醒来”的意思，所以不需要闹钟来叫醒。


========== store[4] ==========
names